# 06 · Tarea 3 — AutoML con Keras Tuner

**Objetivo.** Buscar automáticamente la **topología** de la red de impago con **Keras Tuner** (estrategia **Hyperband** o *RandomSearch*, **D-3.1**) sobre un espacio de hiperparámetros que incluye **nº de capas**, **unidades por capa**, **dropout + su `rate` (sí o sí)**, **learning rate** (escala log), **función de activación** y **`λ_fair`** (**D-3.2**). El `objective` del tuner es **`val_auc`** a maximizar (**D-3.4**), tratando el **fairness como eje externo** y no como objetivo del escalar.

Como el tuner optimiza un único escalar, la frontera de Pareto **no sale gratis**: se construye con un **bucle externo sobre `λ`** (**D-3.3**) que, para cada valor, reentrena/busca la mejor topología y registra el par **(AUC, group gap)** → eso produce el entregable **E2** (Pareto Precisión vs Dependencia FAIR).

El `build_model(hp)` integra la **capa custom** del notebook 04 (ratio + saturación `x^p`) y la **FAIR loss** `BCE + λ·D` del notebook 05; la **mejor topología con dropout** que aquí se fija la **hereda el notebook 07** para **MC-Dropout** (cruce **D-3.2 ↔ D-4.1**). Aguas arriba depende de 03 (baseline de precisión), 04 (capa) y 05 (loss).

## Decisiones a tomar antes de empezar

> Fichas de `docs/DECISIONES.md` para esta tarea. **Estado real** copiado tal cual.
> Las decisiones en **Propuesta**/**Abierta** se **validan con el grupo ANTES de
> implementar**: este notebook asume la *Propuesta* por defecto, pero es revisable.

| Decisión | Opciones | Estado |
|---|---|---|
| **D-3.1** · Estrategia de búsqueda | RandomSearch / **Hyperband** / BayesianOptimization | Propuesta |
| **D-3.2** · Hiperparámetros del espacio | nº capas / unidades / **dropout (+rate)** / lr (log) / activación / λ_fair | Propuesta |
| **D-3.3** · Extraer pares (precisión, dependencia) | callback / **bucle externo sobre λ** / tuner multiobjetivo | Propuesta |
| **D-3.4** · Métrica objetivo del tuner | val_loss / **val_auc** / custom combinada | Propuesta |

> **Cruce dropout 3↔4 (D-3.2 ↔ D-4.1).** El **dropout entra obligatoriamente** en el espacio de búsqueda **porque** el notebook 07 reutiliza esta misma topología para **MC-Dropout** (T pasadas con `training=True` en inferencia → `Var[p]`). Es decir, **07 hereda la red con dropout** que aquí se fija; no introduce un dropout nuevo desacoplado.

> **Fairness como eje externo.** El tuner **NO optimiza fairness**: su `objective` es `val_auc` (D-3.4). El **group gap** (D-2.3, línea base EDA **+3,14 pp**) se recoge en el **bucle externo sobre `λ`** (D-3.3) para trazar la Pareto. **Validar estas decisiones con el grupo antes de codificar.**

## Setup e imports

> **Aviso de instalación.** Este notebook usa **`keras-tuner`**, que **NO está
> instalado por defecto** en el entorno (sí lo están `keras` 3.x y `tensorflow`).
> Antes de lanzar la búsqueda del tuner hay que instalarlo:
>
> ```bash
> pip install keras-tuner --upgrade
> ```
>
> El import va **guardado con `try/except`** para que el esqueleto no se rompa
> aunque la librería falte; las celdas de búsqueda (`# TODO`) solo se ejecutan
> tras instalarla.

In [1]:
# === Setup comun (notebooks de modelado 03-07) ===
import os
os.environ["KERAS_BACKEND"] = "tensorflow"   # backend unico para todo el grupo

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibilidad
RNG = 42
np.random.seed(RNG)
import random; random.seed(RNG)
try:
    import keras
    keras.utils.set_random_seed(RNG)
except Exception:
    pass

# Estilo heredado del EDA / preprocesado
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
COLOR_PAGA   = "#2c7fb8"   # TARGET=0  (paga)
COLOR_IMPAGA = "#d7301f"   # TARGET=1  (impaga)
COLOR_ACENTO = "#41ab5d"   # neutro

# Rutas estandar
PROC_DIR = Path("../data/processed")
FIG_DIR  = Path("../results/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR  = Path("../results/tables");  TAB_DIR.mkdir(parents=True, exist_ok=True)

# --- Especifico de la Tarea 3 (Keras Tuner) ---
import sys; sys.path.append("..")   # para importar src/ (p.ej. src/tuning.py)
import keras

# Import guardado: keras-tuner NO esta instalado por defecto (ver nota previa)
try:
    import keras_tuner as kt
    _HAS_KT = True
except ImportError:
    _HAS_KT = False
    print("keras-tuner no instalado -> pip install keras-tuner --upgrade")

keras-tuner no instalado -> pip install keras-tuner --upgrade


In [2]:
import json
from pathlib import Path
import pandas as pd

# --- Rutas y metadatos (fuente de verdad: metadata.json del preprocesado) ---
PROC_DIR   = Path("../data/processed")                       # relativo a notebooks/
META       = json.loads((PROC_DIR / "metadata.json").read_text(encoding="utf-8"))
FEATURES_X = META["columns"]["features_X"]   # 13 features, en orden
SENSIBLE   = META["columns"]["sensible"]     # "CODE_GENDER"  (s)
TARGET     = META["columns"]["target"]       # "TARGET"       (y)

def cargar_split(nombre):
    """Devuelve (X, y, s) para 'train' | 'val' | 'test'.
    X = DataFrame solo con las 13 features (SIN genero).
    y = Series TARGET (1=impaga, 0=paga).  s = Series CODE_GENDER (M=1/F=0).
    """
    df = pd.read_parquet(PROC_DIR / f"{nombre}.parquet")
    X = df[FEATURES_X]          # input del modelo: el genero NUNCA entra aqui
    y = df[TARGET]
    s = df[SENSIBLE]
    assert SENSIBLE not in X.columns, "FUGA: el genero esta dentro de X"
    return X, y, s

# Materializar los tres cortes
X_train, y_train, s_train = cargar_split("train")
X_val,   y_val,   s_val   = cargar_split("val")
X_test,  y_test,  s_test  = cargar_split("test")

# Nota: convertir a tensores con X_*.to_numpy(dtype="float32") (idem y_*, s_*) en el
# punto de uso; s = CODE_GENDER NUNCA entra en X (solo FAIR loss + auditoria de equidad).

# Resumen de control
print(f"{'split':<7}{'X (filas, cols)':>20}{'y':>12}{'s':>12}{'tasa_impago':>14}")
for n, (X, y, s) in {"train": (X_train, y_train, s_train),
                     "val":   (X_val,   y_val,   s_val),
                     "test":  (X_test,  y_test,  s_test)}.items():
    print(f"{n:<7}{str(tuple(X.shape)):>20}{str(tuple(y.shape)):>12}"
          f"{str(tuple(s.shape)):>12}{y.mean():>14.4%}")

split       X (filas, cols)           y           s   tasa_impago
train          (215254, 13)   (215254,)   (215254,)       8.0728%
val             (46126, 13)    (46126,)    (46126,)       8.0735%
test            (46127, 13)    (46127,)    (46127,)       8.0734%


## Espacio de búsqueda y `build_model(hp)`

Aquí se definirá la función `build_model(hp)` (en `src/tuning.py`) que declara el **espacio de búsqueda** con la API de Keras Tuner (D-3.2): nº de capas, unidades, dropout + `rate`, learning rate (log), activación y `λ_fair`, integrando la **capa custom** (04) al principio y compilando con la **FAIR loss** `BCE + λ·D` (05) y `metrics=[AUC]`.

In [3]:
# TODO: build_model(hp) en src/tuning.py -- hp.Int capas/unidades, hp.Boolean('dropout')+hp.Float('rate'),
#       hp.Float('lr', sampling='log'), hp.Choice('activation'), lambda_fair (hp.Float/Choice);
#       integra la capa custom (04) al principio, compila con FAIR loss BCE + lambda*D (05)
#       + metrics=[keras.metrics.AUC(name='auc')], salida Dense(1, activation='sigmoid'). (D-3.2)

## Búsqueda con el tuner

Aquí se instanciará el tuner **Hyperband** (o *RandomSearch* como base, D-3.1) con `objective=val_auc` a maximizar (D-3.4) y se lanzará `tuner.search(...)` validando en `(X_val, y_val)` sin tocar el test.

In [4]:
# TODO: tuner = kt.Hyperband(build_model, objective=kt.Objective('val_auc', 'max'), ...)
#       (o kt.RandomSearch, D-3.1); tuner.search(X_train_np, y_train_np, validation_data=(X_val_np, y_val_np), ...)
#       (requiere keras-tuner instalado: _HAS_KT == True)

## Bucle externo sobre `λ` (frontera de Pareto)

Aquí se recorrerá una rejilla de valores de **`λ_fair`** (D-3.3) buscando/entrenando la mejor topología en cada uno y guardando la terna **(λ, AUC_val, group_gap)**, materia prima de la Pareto que el `objective` escalar no entrega.

In [5]:
# TODO: for lam in LAMBDAS: buscar/entrenar la mejor topologia para ese lambda y guardar
#       (lam, AUC_val, group_gap = mean(y_hat|M) - mean(y_hat|F)) en una lista de puntos. (D-3.3)

## Construcción de la Pareto AUC vs group gap (E2)

Aquí se dibujará el **scatter** con eje **Y = AUC** y eje **X = group gap** (Dependencia FAIR), se marcará el **frente no dominado** y se persistirán figura y tabla del entregable **E2**.

In [6]:
# TODO: scatter eje Y = AUC, eje X = group gap; resaltar el frente de Pareto no dominado;
#       -> results/figures/06_tuner__pareto_auc_vs_gap.png  y  results/tables/06_tuner__pareto_puntos.csv (E2)

## Selección de la mejor topología

Aquí se elegirá la **configuración de compromiso** (necesariamente **con dropout**) que **hereda el notebook 07** para MC-Dropout (cruce **D-3.2 ↔ D-4.1**), persistiendo sus hiperparámetros y el resumen de trials.

In [7]:
# TODO: elegir la config de compromiso (CON dropout) que HEREDA el NB 07 para MC-Dropout (D-3.2 <-> D-4.1);
#       persistir sus hiperparametros (capas/unidades/rate/lr/activacion/lambda) y el resumen de trials
#       -> results/tables/06_tuner__trials.csv

## Curva de loss del mejor trial (E4)

Aquí se graficará el `history` del **mejor trial** para evidenciar convergencia, entregable **E4** (transversal a todos los notebooks que entrenan un modelo final).

In [8]:
# TODO: graficar history (loss train vs val) del mejor trial
#       -> results/figures/06_tuner__curva_loss_mejor.png (E4)

## Entregables y dependencias

Este notebook produce:

- **E2** · Pareto **AUC vs group gap** → figura `results/figures/06_tuner__pareto_auc_vs_gap.png` + tabla `results/tables/06_tuner__pareto_puntos.csv`.
- **Trials** del tuner → `results/tables/06_tuner__trials.csv` (con la topología de compromiso elegida).
- **E4** · Curva de loss del mejor trial → `results/figures/06_tuner__curva_loss_mejor.png`.
- *(Presentación)* **análisis de la Pareto**: lectura del compromiso precisión ↔ dependencia FAIR.

**Dependencias.** Aguas arriba: **03** (baseline de precisión), **04** (capa custom) y **05** (FAIR loss `BCE + λ·D`), que el `build_model(hp)` integra. Aguas abajo: **07** **hereda la topología + dropout** que aquí se fija para MC-Dropout — el **cruce D-3.2 ↔ D-4.1** que obliga a meter el dropout en el espacio de búsqueda sí o sí.